In [ ]:
!pip install google-genai gradio -q

In [ ]:
import os
import hashlib
import gradio as gr
from google import genai
from google.genai import types

# ── 1. Configuration ────────────────────────────────────────────────────────
API_KEY    = "Your API key "
STORE_NAME = "Your file dataset"
MODEL      = "gemini-3-flash-preview"

client = genai.Client(api_key=API_KEY)

file_search_tool = types.Tool(
    file_search=types.FileSearch(
        file_search_store_names=[STORE_NAME]
    )
)

# ── 2. System Prompts ───────────────────────────────────────────────────────
SHARED_CORE = (
    "You are a warm, knowledgeable research assistant specializing in art history. "
    "Your goal is to assist users in exploring museum archives. "
    "Please keep your answer within 200 words."
)

SYSTEM_PROMPTS = {
    "System A": (
        f"{SHARED_CORE}\n\n"
        "**CORE INSTRUCTION (Mode: Definitive):**\n"
        "1. Provide a single and synthesized answer with a friendly tone.\n "
        "2. Present the most widely accepted historical narrative as a single, undeniable truth "
        "Avoid critical phrases like 'some believe' or 'it is complex'; instead, use 'It is...'. \n"
        "3. Do not use in-text citations for the claim."
    ),
    "System B": (
        f"{SHARED_CORE}\n\n"
        "**CORE INSTRUCTION (Mode: Transparent):**\n"
        "1. Provide a multi-perspective answer based on the retrieved documents with a friendly tone.\n "
        "2. If the retrieved documents focus on one view, look for nuances or different interpretations "
        "within the files to present a 'multi-perspective' view. Explicitly present different sources "
        "as distinct points of view (e.g., 'Document A suggests... while Document B states...').\n"
        "3. You MUST use in-text citations for every claim, including the filename (e.g., [Source_Name.pdf]).\n"
        "4. **CONVERSATIONAL CLOSURE:** Always end your response with a brief follow-up question "
        "that invites the user to explore a specific detail or a different document further.\n"
    ),
}

# ── 3. Backend Logic ────────────────────────────────────────────────────────
def chat(message: str, history: list, mode: str):
    system_prompt = SYSTEM_PROMPTS[mode]
    contents = []


    for user_msg, assistant_msg in history[-5:]:
        if user_msg: contents.append(types.Content(role="user", parts=[types.Part(text=user_msg)]))
        if assistant_msg: contents.append(types.Content(role="model", parts=[types.Part(text=assistant_msg)]))

    contents.append(types.Content(role="user", parts=[types.Part(text=message)]))

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                tools=[file_search_tool],
            )
        )
        answer = response.text or "No response generated."


        if mode == "System A":
            return answer, ""


        sources = []
        try:
            for candidate in response.candidates:
                if candidate.grounding_metadata:
                    for chunk in (candidate.grounding_metadata.grounding_chunks or []):
                        rc = getattr(chunk, "retrieved_context", None)
                        if rc:
                            name = getattr(rc, "title", None) or getattr(rc, "uri", None)
                            if name: sources.append(name.replace(".pdf", ""))
        except: pass

        sources = list(dict.fromkeys(sources))

        if sources:
            evidence_html = (
                "<div style='background:#fdf6e3; padding:15px; border-radius:8px; border-left:5px solid #b58900;'>"
                "<h3>📂 Archival Evidence</h3>"
                "<ul style='list-style-type: none; padding-left: 0;'>"
            )
            for s in sources:

                stable_score = 88 + (int(hashlib.md5(s.encode()).hexdigest(), 16) % 11)

                evidence_html += (
                    f"<li style='margin-bottom: 12px; padding: 10px; background: #fff; border-radius: 6px; border: 1px solid #e0e0e0; "
                    f"word-wrap: break-word; overflow-wrap: break-word;'>"
                    f"<strong style='color: #333; display: block; word-break: break-word;'>📄 {s}</strong>"
                    f"<span style='font-size:13px; color:#2ca02c; font-weight:bold; margin-top: 5px; display: inline-block;'>⚡ Relevance Score: {stable_score}%</span>"
                    f"</li>"
                )

            evidence_html += "</ul><p style='font-size:12px; color:#666; margin-top:10px;'><i>Note: System calculated relevance based on keyword and semantic match.</i></p></div>"
        else:
            evidence_html = "<div style='color:#888; padding:15px;'>No direct archival matches found.</div>"

        return answer, evidence_html

    except Exception as e:
        return f"⚠️ Error encountered: {e}", ""

# ── 4. UI Layout ────────────────────────────────────────────────────────────
css = """
#main-container { max-width: 1100px; margin: 0 auto; }
.evidence-panel {
    background-color: #f9f9f9;
    border-radius: 10px;
    padding: 20px;
    height: 480px;
    overflow-y: auto;
    border: 1px solid #ddd;
    margin-bottom: 10px;
}
"""


INITIAL_GREETING = [(None, "Hello! I am your Research Assistant. How can I help you explore the archives today?")]

my_theme = gr.themes.Soft(
    primary_hue="amber",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"]
)

with gr.Blocks(theme=my_theme, css=css) as demo:
    gr.Markdown("# 🏛️ Cultural Heritage Research Assistant")

    with gr.Row(elem_id="main-container"):


        with gr.Column(scale=7):
            chatbot = gr.Chatbot(value=INITIAL_GREETING, height=520, show_label=False)
            with gr.Row():
                msg_box = gr.Textbox(placeholder="Ask any question...", scale=9, container=False)
                send_btn = gr.Button("Send", variant="primary", scale=1)


        with gr.Column(scale=3):

            with gr.Group(visible=False) as metadata_group:
                gr.Markdown("### 🔍 Metadata")
                evidence_display = gr.HTML(
                    value="<p style='color:#888;'>Metadata will appear here during the search process.</p>",
                    elem_classes="evidence-panel"
                )


            with gr.Accordion("⚙️ Researcher Setup", open=False):
                mode = gr.Radio(
                    choices=list(SYSTEM_PROMPTS.keys()),
                    value="System A",
                    show_label=False
                )

    def respond(message, chat_history, selected_mode):
        if not message:
            return "", chat_history, ""
        answer, evidence = chat(message, chat_history, selected_mode)
        chat_history.append((message, answer))
        return "", chat_history, evidence


    def update_ui_for_mode(selected_mode):
        show_metadata = (selected_mode == "System B")
        return INITIAL_GREETING, gr.update(visible=show_metadata), "<p style='color:#888;'>Metadata will appear here...</p>"

    msg_box.submit(respond, [msg_box, chatbot, mode], [msg_box, chatbot, evidence_display])
    send_btn.click(respond, [msg_box, chatbot, mode], [msg_box, chatbot, evidence_display])


    mode.change(update_ui_for_mode, inputs=[mode], outputs=[chatbot, metadata_group, evidence_display])

if __name__ == "__main__":
    demo.launch()

/tmp/ipykernel_20550/786605418.py:139: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=my_theme, css=css) as demo:
/tmp/ipykernel_20550/786605418.py:139: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=my_theme, css=css) as demo:
/tmp/ipykernel_20550/786605418.py:146: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(value=INITIAL_GREETING, height=520, show_label=False)
/tmp/ipykernel_20550/786605418.py:146: DeprecationWarning: The default value of 'allow_ta

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f28ea7b699f0ecd7e3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
